### Tarea #3 Gema Guerra Valdez (1819110)

#### Requisitos

- Hacer DOE para comparar modelos y sus hiperparámetros con relación a la clasificación de textos

- Escribir un reporte con los hallazgos, metodología y resultados en PDF y subirlo en una sección claramente identificable de tu repositorio.

In [1]:
pip install kaggle

In [2]:
import os
import zipfile
import pandas as pd
import kagglehub

link to dataset: https://www.kaggle.com/datasets/tobiasbueck/capterra-reviews

In [3]:
path = kagglehub.dataset_download("tobiasbueck/capterra-reviews")

print("Path to dataset files:", path)

100%|██████████| 0.98M/0.98M [00:00<00:00, 78.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/tobiasbueck/capterra-reviews/versions/5


In [4]:
archivos = os.listdir(path)
print("Archivos encontrados:", archivos)
df = pd.read_csv(path + "/" + archivos[0])
df.head(5)

Archivos encontrados: ['capterra_reviews.csv']


,ticket_system,title,overall_text,pros_text,cons_text,overall_rating,ease_of_use,customer_service,features,value_for_money,...,Ticket Creation and Assignment,Automated Ticket Routing,Status Tracking and Updates,Priority and SLA Management,Customer and Agent Portals,Knowledge Base Integration,Email Notifications and Alerts,Reporting and Analytics,Customizable Workflows,"Multi-Channel Support (Email, Chat, Phone)"
0,Zoho Desk,Excellent solution that meets all of our requi...,Zoho Desk is a top-tier platform for developin...,"As a ticketing and customer service platform, ...",Although the program provides a great return o...,5,5.0,5.0,5.0,4.0,...,1,-1,1,1,0,0,0,1,0,0
1,Zoho Desk,Offers multiple options to help customers get ...,"Provides web-based customer support, reducing ...","In the period I have used Zoho Desk, I have ex...",Zoho Desk gives maximum scalability and return...,5,5.0,5.0,5.0,5.0,...,1,0,0,0,0,1,0,0,1,0
2,Zoho Desk,"Zoho Desk isn't a favourite option of mine, bu...","Zoho Desk is a great tool with many features, ...",Zoho Desk offers a range of tools to make sure...,Unfortunately the creation and customisation o...,3,3.0,4.0,3.0,3.0,...,1,0,1,0,1,1,0,0,-1,1
3,Zoho Desk,Keep your customers happy,"Zoho Desk is very responsive and fast, is pack...",In my line of business the returning customers...,We have experienced slow loading some time ago...,5,5.0,5.0,5.0,5.0,...,1,0,-1,0,1,1,0,0,0,1
4,Zoho Desk,A fantastic tool for answering customer queries,"Questions about orders or invoices, tickets, a...",Managing a staff to answer client questions an...,"Getting help, including clear responses to my ...",4,5.0,4.0,5.0,5.0,...,1,0,0,0,1,0,-1,0,1,0


In [5]:
import os
import re
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

In [6]:
path = kagglehub.dataset_download("tobiasbueck/capterra-reviews")
csv_filename = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_filename))
len(df)

Using Colab cache for faster access to the 'capterra-reviews' dataset.


4899

### ETL

In [7]:
col_texto = 'overall_text'
col_target = 'Automated Ticket Routing'

# Limpieza
df = df.dropna(subset=[col_texto, col_target])

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)  # Remover caracteres especiales
    texto = re.sub(r'\s+', ' ', texto).strip()  # Eliminar espacios dobles
    return texto

df['clean_text'] = df[col_texto].apply(limpiar_texto)

X = df['clean_text']
y = df[col_target].astype(str)  # Convierte -1, 0, 1 a categorías textuales

### DOE

In [8]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2500, stop_words='english')),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
])

# Definición de los Factores y Niveles
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],  # Factor A: Unigramas vs Bigramas
    'rf__n_estimators': [50, 150]            # Factor B: 50 vs 150 árboles
}

print("Calculando 3 réplicas por tratamiento mediante 3-Fold Cross Validation...")

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    return_train_score=False
)
grid_search.fit(X, y)


Ejecutando DOE
Calculando 3 réplicas por tratamiento mediante 3-Fold Cross Validation...


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(max_features=2500,
                                                        stop_words='english')),
                                       ('rf',
                                        RandomForestClassifier(n_jobs=-1,
                                                               random_state=42))]),
             param_grid={'rf__n_estimators': [50, 150],
                         'tfidf__ngram_range': [(1, 1), (1, 2)]},
             scoring='f1_macro')

In [10]:
resultados = grid_search.cv_results_

# Extraemos los valores de los factores convirtiéndolos de forma segura a texto plano
factor_a_text = [str(val) for val in resultados['param_tfidf__ngram_range']]
factor_b_text = [str(val) for val in resultados['param_rf__n_estimators']]

doe_table = pd.DataFrame({
    'Tratamiento': [1, 2, 3, 4],
    'Factor_A (N-grams)': factor_a_text,
    'Factor_B (Estimators)': factor_b_text,
    'F1_Rplica_1': resultados['split0_test_score'],
    'F1_Rplica_2': resultados['split1_test_score'],
    'F1_Rplica_3': resultados['split2_test_score'],
    'F1_Promedio (Y)': resultados['mean_test_score'],
    'Error_Std': resultados['std_test_score']
})

print("\n================ TABLA DE RESULTADOS DEL DOE ================")
print(doe_table.to_string(index=False))

print("\nConfiguración óptima según el experimento:")
print(grid_search.best_params_)

{'mean_fit_time': array([2.05431938, 1.86781351, 6.01071533, 5.49435663]), 'std_fit_time': array([0.48378276, 0.06550113, 0.15802728, 0.49492452]), 'mean_score_time': array([0.13221129, 0.13053838, 0.22667289, 0.27034775]), 'std_score_time': array([0.03674826, 0.00280741, 0.0081411 , 0.02915772]), 'param_rf__n_estimators': masked_array(data=[50, 50, 150, 150],
             mask=[False, False, False, False],
       fill_value=999999), 'param_tfidf__ngram_range': masked_array(data=[(1, 1), (1, 2), (1, 1), (1, 2)],
             mask=[False, False, False, False],
       fill_value=np.str_('?'),
            dtype=object), 'params': [{'rf__n_estimators': 50, 'tfidf__ngram_range': (1, 1)}, {'rf__n_estimators': 50, 'tfidf__ngram_range': (1, 2)}, {'rf__n_estimators': 150, 'tfidf__ngram_range': (1, 1)}, {'rf__n_estimators': 150, 'tfidf__ngram_range': (1, 2)}], 'split0_test_score': array([0.33519259, 0.35892101, 0.34217377, 0.34778881]), 'split1_test_score': array([0.35396185, 0.36977583, 0.35806